# Backfilling Data Transactions iSeller Mitra

>An ETL proccess for backfilling raw data transactions from API 'https://janjijiwamitra.isellershop.com/api/v2/GetOrders' 

> Updated at: 2026-10-28

> Created by: irpanputra@jiwagroup.com

> Updated by: lahiardhan@jiwagroup.com

LATEST UPDATE :
> - 2026-10-13 : Fix Accuracy Details vs Headers (Discount/Promo Fan-out, Refund, Combo Allocation, Tax from API)
> - 2026-10-13 : Enhance API Fetching Completeness Check (Retry on API Error, Re-fetch Empty Response, Compare vs Data Lake)
> - 2026-10-16 : Fix Custom Discount (Kasir) Not Deducted on Details -> Use discounts[] Amount
> - 2026-10-16 : Fix Missing Orders on Window Boundary (xx:59:59) -> Window End at xx:59:59.999
> - 2026-10-16 : Simplify API Fetching Cell (Keep Retry, Re-fetch Empty & Per-Outlet Progress)
> - 2026-10-28 : Fix Backfilling Flow (Only Insert Orders Missing in Data Lake, Combo/Promo/Discount Parsing, Secrets from .env)
> - 2026-10-28 : Enhance API Fetching Resilience (5 Retries, Failed Calls Re-fetched at the End, Partial Run Still Dumps Successful Data in 1 Transaction then Fails Loudly)


# IMPORT LIBRARY

In [ ]:
import datetime
import pymysql
import time
from pytz import timezone

import requests
import json
from dateutil import parser

import pandas as pd
import numpy as np 
import importlib, importlib.util

import warnings
warnings.filterwarnings('ignore')

from sqlalchemy import create_engine
import psycopg2

import mysql.connector
from mysql.connector import Error
from mysql.connector import errorcode

# CONFIG DB

In [ ]:
# Import Config Data
import importlib, importlib.util

def module_from_file(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

config_dl_dwh = module_from_file("config_dw", "../../config/config_db_datawarehouse.py")

# FUNCTION

In [ ]:
def read_transaction_id(start_date, end_date):
    connection = mysql.connector.connect(host=config_dl_dwh.hostname,
                             port=3306,
                             user=config_dl_dwh.username,
                             password=config_dl_dwh.password,
                             db=config_dl_dwh.db_resource)

    cursor=connection.cursor()

    sql = """
            select 
                distinct order_id
            from transactions_iseller_mitra 
            where str_to_date(transaction_date, '%Y-%m-%d') between '{0}' and '{1}'
          """.format(start_date, end_date)
    cursor.execute(sql)

    record = cursor.fetchall()
    list_record = list(record)
    
    return list_record

In [ ]:
def get_existing_order_id(list_order_id):
    connection = pymysql.connect(host=config_dl_dwh.hostname,
                             port=3306,
                             user=config_dl_dwh.username,
                             password=config_dl_dwh.password,
                             db=config_dl_dwh.db_resource)

    cursor=connection.cursor()
    
    list_order_id = tuple(list_order_id)
    
    params = {'list_order_id' : list_order_id}

    sql ="""
            SELECT 
                DISTINCT order_id
            FROM transactions_items_iseller_mitra
            WHERE order_id IN %(list_order_id)s
        """
    
    cursor.execute(sql,params)

    record = cursor.fetchall()
    list_record = list(record)
    
    return list_record

In [ ]:
def expand_nested_lists(df, column_names):
    expanded_data = []
    for index, row in df.iterrows():
        for column_name in column_names:
            data_list = row[column_name]
            if isinstance(data_list, list): 
                for item in data_list:
                    if item is not None:  
                        new_row = row.copy() 
                        for key, value in item.items():
                            new_column_name = f"{column_name}_{key}"  
                            new_row[new_column_name] = value
                        expanded_data.append(new_row)
            else:
                expanded_data.append(row)
    df_expanded = pd.DataFrame(expanded_data)
    return df_expanded

In [ ]:
def convert_data_to_dataframe(data):
    
    df_all = pd.DataFrame(data)

    for column in ['transactions']:
        df_all[column] = df_all[column].apply(lambda x: x[0] if x else None)  
        if df_all[column][0] is not None: 
            if isinstance(df_all[column][0], dict): 
                for key in df_all[column][0].keys():
                    new_column = f"{column}_{key}"
                    df_all[new_column] = df_all[column].apply(lambda x: x[key] if x else None)

    return df_all

In [ ]:
def adjust_timezone(date_string, hours_offset):
    datetime_obj = datetime.datetime.strptime(date_string, "%Y-%m-%d %H:%M:%S")
    adjusted_datetime = datetime_obj + datetime.timedelta(hours=hours_offset)
    return adjusted_datetime.strftime("%Y-%m-%d %H:%M:%S")

# Main Progress

### 1. Get Auth Token and Date List

In [ ]:
import os
try:
    from dotenv import load_dotenv
    load_dotenv('../../.env')
except ImportError:
    pass

# Token dibaca dari .env / environment (JANGAN hardcode token di notebook)
access_token = (os.environ.get('access_token_mitra') or '').strip()
assert access_token, "access_token_mitra belum di-set di .env / environment"


In [ ]:
today = datetime.date.today() - datetime.timedelta(days=8)
today_string = today.strftime("%Y-%m-%d")

timezone_offset_hours = -7

start_date_list = []
end_date_list = []

#looping 2 days
for day_offset in range(2):
    for i in range(6):
        start_hour = i * 4
        end_hour = (i + 1) * 4 - 1

        start_date_string = (today + datetime.timedelta(days=day_offset)).strftime("%Y-%m-%d") + f" {start_hour:02}:00:00"
        adjusted_start_date = adjust_timezone(start_date_string, timezone_offset_hours)
        start_date_list.append(adjusted_start_date)

        end_date_string = (today + datetime.timedelta(days=day_offset)).strftime("%Y-%m-%d") + f" {end_hour:02}:59:59"
        adjusted_end_date = adjust_timezone(end_date_string, timezone_offset_hours)
        end_date_list.append(adjusted_end_date + ".999")

print(f"start date list: {start_date_list}")
print(f"end date list : {end_date_list}")


In [ ]:
start_date = datetime.date.today() - datetime.timedelta(days=8)
end_date = datetime.date.today() - datetime.timedelta(days=6)

start_date_string = start_date.strftime("%Y-%m-%d")
end_date_string = end_date.strftime("%Y-%m-%d")

### 2. Get List Transaction from DB

In [ ]:
list_order_id = []

start_date = start_date_string
end_date = end_date_string

data = read_transaction_id(start_date, end_date)
df_transaction = pd.DataFrame(data)
                        
for p in range(0, len(df_transaction)):
    list_order_id.append(df_transaction[0][p])

In [ ]:
len(list_order_id)

### 3. Get Store Information

In [ ]:
url = 'https://janjijiwamitra.isellershop.com/api/v2/getstoreinfo'

headers = {
    'Authorization': f'Bearer {access_token}',
    'Content-Type': 'application/x-www-form-urlencoded'
}

process = requests.get(url, headers=headers)

string_to_json = json.loads(process.text)
df = pd.DataFrame(list(string_to_json.items()),columns = ['name','value'])

In [ ]:
x = df['value'][0]

df_x = pd.DataFrame(list(x.items()),columns = ['name','value'])
outlet_data = df_x['value'][0]

df_outlet = pd.DataFrame(outlet_data)

df_outlet['outlet_code'] = df_outlet['outlet_code'].str.zfill(5)

df_outlet = df_outlet[df_outlet['outlet_code'] != 'Main Outlet' ]

list_of_outlets = df_outlet['outlet_id'].unique()

In [ ]:
len(list_of_outlets)

### 4. Get Item Information

In [ ]:
url = 'https://janjijiwamitra.isellershop.com/api/v2/GetProducts'

headers = {
    'Authorization': f'Bearer {access_token}',
    'Content-Type': 'application/x-www-form-urlencoded'
}

flag = 0
df_product_data = pd.DataFrame() 
get_product = []

for i in range(1, 5): #karena saat ini total products dan bundling kurang lebih hanya 1200 maka buat looping 4

    print(flag)

    data = {
    'modified_after': '2023-01-01',
    'track_inventory': 'FALSE',
    'page_size' : 500, #max pages 
    'page' : i
    }

    process = requests.post(url, headers=headers, data=data)
    string_to_json = json.loads(process.text)
    df = pd.DataFrame(list(string_to_json.items()), columns=['name', 'value']) 
    try:
        product_data = pd.DataFrame(df['value'][0])
        df_product_data = pd.concat([product_data, df_product_data], ignore_index=False) 
        get_product.append(df['value'][0])
    except:
        pass

    flag += 1

print(f"Total Master Data : {len(df_product_data)}")

In [ ]:
#membagi master data products (per-sku) dan bundling (per bundling_code)
 
df_bundlings = df_product_data[df_product_data['type'] == 'comboset'] #data bundling
df_products = df_product_data[df_product_data['type'] != 'comboset'] #data products 

#normalize master data products
df_products = df_products[
    [
        'product_id', 'product_type', 'sku', 'name', 'type', 'taxable', 'price', 'sold_count', 'modified_date'
    ]
]
df_products_anomaly = df_products[(df_products['sku'] == '') | (df_products['price'] == 0)]
df_products =  df_products[df_products['sku'] != '']
df_products = df_products[df_products['price'] != 0]

#normalize master data bundlings
df_bundlings = df_bundlings[['product_id', 'name', 'sku', 'taxable', 'sold_count', 'bundles', 'modified_date']]
df_bundlings.columns = ['bundling_id', 'bundling_name', 'bundling_code', 'taxable', 'sold_count', 'bundles', 'modified_date']
columns_to_expand = ['bundles']
df_bundling_exp = expand_nested_lists(df_bundlings, columns_to_expand)
x_columns = [col for col in df_bundling_exp.columns if col.startswith('bundles_')]
for col in x_columns:
    new_col_name = col.replace('bundles_', '')
    df_bundling_exp.rename(columns={col: new_col_name}, inplace=True)
df_bundling_exp.drop('bundles', axis=1, inplace=True)
df_bundling_final = pd.merge(df_bundling_exp, df_products[['product_id', 'price']], on='product_id', how='left')
df_bundlings = df_bundling_final[
    [
        'bundling_id', 'bundling_name', 'bundling_code', 'taxable', 'product_id', 'sku', 
        'category', 'product_name', 'variant_name', 'quantity', 'price', 'taxable', 
        'sold_count', 'modified_date'
    ]
]
df_bundlings['bundling_code'] = df_bundlings.apply(lambda row: row['bundling_code'][:14] if pd.notnull(row['bundling_code']) else None, axis=1)

print(f"Total Master Data Product : {len(df_products)}")
print(f"Total Master Data Bundling : {df_bundlings['bundling_id'].nunique()}")
print(f"Total Product dengan SKU Null : {len(df_products[df_products['sku'].isna()])}")

### 5. Get Order Data

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import time

url = 'https://janjijiwamitra.isellershop.com/api/v2/GetOrders'

headers = {
    'Authorization': f'Bearer {access_token}',
    'Content-Type': 'application/x-www-form-urlencoded'
}

API_MAX_RETRIES = 5   # jeda antar percobaan 2, 4, 8, 16 detik (error sementara iSeller, mis. deadlock, biasanya lewat)

# 1 task = 1 outlet x 1 jendela waktu
tasks = [
    (outlet_id, start_date_list[i], end_date_list[i])
    for i in range(len(start_date_list))
    for outlet_id in list_of_outlets
]
print(f"Total API calls: {len(tasks)}")


def fetch_orders(task):
    outlet_id, start_dt, end_dt = task
    data = {
        'created_after': start_dt,
        'created_before': end_dt,
        'timezone': '7',
        'includes': 'orderdetails, promotiondetails, transactions, discountdetails',
        'outlet_id': outlet_id
    }
    for attempt in range(API_MAX_RETRIES):
        try:
            result = requests.post(url, headers=headers, data=data, timeout=30).json()
            # API bisa balas 200 tapi isinya error (rate limit, token, dll.) -> anggap gagal, coba lagi
            if result.get('status') is False or result.get('error_message'):
                raise ValueError(result.get('error_message'))
            return list(result.values())[1]
        except Exception as e:
            if attempt == API_MAX_RETRIES - 1:
                raise Exception(f"GAGAL outlet={outlet_id} {start_dt}~{end_dt}: {e}")
            time.sleep(2 ** (attempt + 1))


# get_data[i]: None = belum berhasil ditarik, [] = berhasil tapi 0 order
get_data = [None] * len(tasks)
calls_per_outlet = len(start_date_list)
outlet_done = {outlet_id: 0 for outlet_id in list_of_outlets}
n_outlet_selesai = 0


def run_calls(indices, workers, desc, track_outlets=False):
    """Tarik panggilan `indices` paralel. Panggilan yang tetap gagal TIDAK menghentikan run: dikembalikan sebagai {index: pesan error}."""
    global n_outlet_selesai
    failed = {}
    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(fetch_orders, tasks[i]): i for i in indices}
        for future in tqdm(as_completed(futures), total=len(futures), desc=desc):
            i = futures[future]
            try:
                get_data[i] = future.result()
            except Exception as e:
                failed[i] = str(e)

            if track_outlets:
                outlet_id = tasks[i][0]
                outlet_done[outlet_id] += 1
                if outlet_done[outlet_id] == calls_per_outlet:
                    n_outlet_selesai += 1
                    print(f"  [outlet done] {outlet_id} ({n_outlet_selesai}/{len(list_of_outlets)} outlets)")
    return failed


# 1) tarik semua
failed = run_calls(range(len(tasks)), 10, "Fetching", track_outlets=True)

# 2) panggilan yang gagal diulang dengan worker lebih sedikit (beban ke iSeller lebih ringan)
if failed:
    print(f"Panggilan gagal: {len(failed)} dari {len(tasks)} -> diulang dengan 3 worker")
    failed = run_calls(sorted(failed), 3, "Re-fetch gagal")

# 3) panggilan yang hasilnya 0 order diulang sekali (bisa jadi tadi gagal diam-diam)
empty_idx = [i for i, orders in enumerate(get_data) if orders is not None and not orders]
print(f"Panggilan dengan 0 order: {len(empty_idx)} dari {len(tasks)} -> diulang sekali")
before_empty = len(empty_idx)
run_calls(empty_idx, 5, "Re-fetch 0 order")   # kalau re-fetch ini gagal, hasil awal ([] 0 order) tetap dipakai
recovered = sum(1 for i in empty_idx if get_data[i])
print(f"Ternyata ada order di {recovered} panggilan (sudah dipakai). Sisa 0 order: {before_empty - recovered}")

# 4) panggilan yang TETAP gagal: run tetap lanjut, tapi di akhir (sel dump) run digagalkan supaya tidak terlewat
failed_calls = [(tasks[i][0], tasks[i][1], tasks[i][2], msg) for i, msg in sorted(failed.items())]
if failed_calls:
    print("\n" + "!" * 70)
    print(f"PERINGATAN: {len(failed_calls)} panggilan API TETAP GAGAL setelah semua retry. Data jendela ini BELUM masuk:")
    for outlet_id, start_dt, end_dt, msg in failed_calls[:30]:
        print(f"  {msg[:200]}")
    print("Run tetap lanjut memproses data yang berhasil. Jalankan ulang notebook ini nanti: hanya order yang belum ada di Data Lake yang akan dimasukkan.")
    print("!" * 70)
else:
    print("Semua panggilan API berhasil.")


In [ ]:
# backfill: ambil hanya order yang BELUM ada di Data Lake (list_order_id dari sel 2)
existing_order_ids = set(list_order_id)
missing_orders = [order for batch in get_data if batch for order in batch if order['order_id'] not in existing_order_ids]
print(f"Order dari API: {sum(len(b) for b in get_data if b):,} | sudah ada di Data Lake: {len(existing_order_ids):,} | perlu di-backfill: {len(missing_orders):,}")

# dibungkus 1 list supaya bentuknya sama dengan ingestion; kalau kosong, cell selanjutnya otomatis "tidak ada data"
get_data = [missing_orders] if missing_orders else []
data = get_data.copy()


### 6. Transform the Data

In [ ]:
if len(get_data) > 0:
    try:
        
        df_headers = pd.DataFrame()
        df_details = pd.DataFrame()
        df_product_combos = pd.DataFrame()
        df_promo = pd.DataFrame()
        df_discount = pd.DataFrame()

        for headers in data:
            if len(headers) > 0:
                headers_list_cols = [
                    'outlet_id', 'order_reference', 'outlet_code', 'outlet_name', 'order_date', 'closed_date', 'status', 'channel', 'channel_detail', 'payment_status', 'fulfillment_status',
                    'notes', 'customer_id', 'customer_first_name', 'customer_last_name', 'customer_phone_number', 'customer_email', 'cashier_id',
                    'total_order_amount', 'total_discount_amount', 'total_promotion_amount', 'subtotal', 'total_tax_amount', 'total_amount'
                ]
                df_headers = pd.concat([df_headers, pd.json_normalize(headers, record_path=['transactions'], meta=headers_list_cols , meta_prefix= 'transactions_')])
                df_details = pd.concat([df_details, pd.json_normalize(headers, record_path=['order_details'])])

                # sama dengan ingestion: order_detail_id diambil dari parent lewat record_path + meta
                for details in headers:
                    if len(details['order_details']) > 0:
                        df_product_combos = pd.concat([df_product_combos, pd.json_normalize(details['order_details'], record_path='product_combos', meta='order_detail_id')])
                        df_promo = pd.concat([df_promo, pd.json_normalize(details['order_details'], record_path='promotions', meta='order_detail_id')])
                        df_discount = pd.concat([df_discount, pd.json_normalize(details['order_details'], record_path='discounts', meta='order_detail_id')])

        # backfill datanya sedikit, bisa jadi tidak ada combo sama sekali -> kolom tetap harus ada untuk merge
        if df_product_combos.empty:
            df_product_combos = pd.DataFrame(columns=['order_detail_id', 'product_id', 'product_sku', 'product_name', 'variant_name', 'quantity'])

    # get data headers
        df_headers = df_headers[['order_id', 'transaction_id', 'transaction_date', 'transactions_order_reference',
                    'transactions_outlet_id', 'transactions_outlet_code', 'transactions_outlet_name',
                    'transactions_channel', 'transactions_channel_detail', 'transactions_order_date',
                    'transactions_closed_date', 'transactions_status', 'transactions_payment_status', 
                    'transactions_fulfillment_status', 'type', 'status',  'gateway', 'payment_type_name', 
                    'transactions_notes', 'transactions_customer_id', 'transactions_customer_first_name', 
                    'transactions_customer_last_name', 'transactions_customer_phone_number', 
                    'transactions_customer_email', 'transactions_cashier_id', 'transactions_total_order_amount',
                    'transactions_total_discount_amount', 'transactions_total_promotion_amount', 'transactions_subtotal',
                    'transactions_total_tax_amount', 'transactions_total_amount', 'mdr']] 

        df_headers.columns = [
                            'order_id','transaction_id', 'transaction_date', 'order_reference', 'outlet_id', 'outlet_code',
                            'outlet_name', 'channel', 'channel_detail','order_date', 'closed_date', 'status', 'payment_status', 'fulfillment_status', 
                            'transactions_type', 'transactions_status', 'transactions_gateway', 'transactions_payment_type_name', 
                            'notes', 'customer_id', 'customer_first_name', 'customer_last_name', 'customer_phone_number','customer_email', 'cashier_id', 
                            'total_order_amount', 'total_discount_amount', 'total_promotion_amount', 'subtotal', 'total_tax_amount', 
                            'total_amount', 'mdr'
                    ]

        df_headers['order_date'] = pd.to_datetime(df_headers['order_date'])
        df_headers['closed_date'] = pd.to_datetime(df_headers['closed_date'])
        df_headers['outlet_code'] = df_headers['outlet_code'].str.zfill(5)

        gmt_plus_7 = timezone('Etc/GMT-7')

        df_headers['transaction_date'] = pd.to_datetime(df_headers['transaction_date'])
        df_headers['transaction_date'] = df_headers['transaction_date'].dt.tz_localize('UTC').dt.tz_convert(gmt_plus_7)

        df_headers['customer_first_name'] = df_headers['customer_first_name'].astype(str).apply(lambda x: x.encode('ascii', 'ignore').decode('ascii'))
        df_headers['customer_last_name'] = df_headers['customer_last_name'].astype(str).apply(lambda x: x.encode('ascii', 'ignore').decode('ascii'))
        df_headers['notes'] = df_headers['notes'].astype(str).apply(lambda x: x.encode('ascii', 'ignore').decode('ascii'))

        #set sales type on each order
        # df_headers_sjw = df_details[['order_id', 'product_id']]

        product_id_sjw_sjo = {
            '1d2664fc-f2c4-4d63-93b0-0150c5bad9ca',
            '514f7f40-fcf1-4b24-a881-0e32a6e2da88',
            'b7a5bd88-cda9-4623-814a-734767026910',
            '134f7a5b-09a2-455c-8f70-93a01468131e',
            'b26ce832-29a4-46f7-be9f-e9f4d72c1dca',
            '64b0cc8b-88e4-498a-b5b5-ff093b5915e5',
            '2306edf4-9f70-4c79-88d9-2a836a292816',  # added at 2024-03-13
            '9eef163d-fd2f-4e52-98e6-4593679a16b5',  # added at 2024-04-01 semua SJO
            '9ea8be5c-8bc1-4f14-b564-5eb75a19d907',
            'a96b7a4e-b86d-4171-956b-77a09d1c54f9',
            '6b07599f-14f8-44f9-ab5b-4389bf7e82d1',
            '328e5672-48a1-4683-b65d-c4a45f344ca1',  # added at 2024-04-02
            'baa1d6c2-7b48-40af-a6f3-b9e481f93e61',
            '3ab4aa05-f664-4eaf-ad7e-bdf643d421d4',
            '2fcb7dad-bd59-44c9-a846-1279f9840d37',
            'c038d0fa-a12a-4644-9bbb-3878b3064912',
            '4521dc51-b3d6-4493-814a-617fde292c04',  # added at 2024-05-14
            '62701d95-4d24-449e-b0db-ea8994d00b62',  # added at 2024-07-08
            '8f5fe78b-7d7c-46ed-9f5b-c9f8d195b60d',  # added at 2024-08-04
            '44d2e817-8745-4cbf-aa36-d4abfc75aebf',
            'a26111f7-3aa5-48b0-a77b-2f952aab5eae',
            '74ee1eae-21b8-467a-828c-b59cf780d065',  # added at 2024-08-16
            'a89893bf-9026-4a5c-9f0c-949c0f4539d7',
            '150c96b0-fe81-4835-b3cc-593bfb6c45db',  # added at 2024-09-14
            '999251a7-5e8b-4267-b67c-b33cedead904',  # added at 2024-10-03
            '0c8a73fe-ae1b-4287-a6af-7d48fce86de5',
            'c9d038f6-fea1-494f-813c-ad0aed0dde1e',  # added at 2024-10-23
            'a94730e0-4f5b-4af9-b70b-68dbea299b78',
            '5842673c-8a8b-423b-9804-1c0d484c60ac',  # added at 2024-11-18
            'bae1b783-e862-4c5f-bb46-b20bb7e02b63',
            '00295e2b-66bb-489f-9615-6e25676c8331',
            'd8a7f659-c1dd-448a-9110-688fd1105bf8',
            '47855957-b056-49bc-8be3-c8ea68e0160b',
            '5864c6dc-216e-4764-9c8b-71bc9d77c41a',  # added at 2025-02-26
            '3ffe31ac-f4c6-4206-959f-26f2ec6224fc',
            '64b5deb0-fcef-412f-bd2e-714a9b65a6b9',  # added at 2025-06-23
            '7ddc0da7-72bf-4518-ac37-c1c62e48d418',  # added at 2026-08-20 (Basreng Pedas Daun Jeruk, ITM010872)
            '7fed141a-8b22-4cee-80c0-df4242e136ec',  # added at 2026-10-09 (Es Kopi Susu Honey, FCH040025)
            '5faac2db-7827-4414-a5ac-1345a79299dd',  # added at 2026-10-09 (Matcha - KSJ, FCH040036)
            '06aa6648-b40a-4d02-b9b7-3105fa83c762'   # added at 2026-10-09 (Matcha - SJO, ITM022191)
        }

        # is_sjw = 1 nilai per order: order berisi minimal 1 produk SJW -> Sejuta Jiwa
        df_headers_sjw = (df_details['product_id'].isin(product_id_sjw_sjo)
                        .groupby(df_details['order_id']).any()
                        .rename('is_sjw').reset_index())
        df_merged = pd.merge(df_headers, df_headers_sjw, on='order_id', how='left')
        df_merged[df_merged['is_sjw'] == True]['transactions_payment_type_name'].unique()

        #create conditions for set sales type (Per 5 Oct 2023)
        conditions = [
            (df_merged['transactions_payment_type_name'] == 'Go Food'),
            (df_merged['transactions_payment_type_name'] == 'Grab Food'),
            (df_merged['transactions_payment_type_name'] == 'Shopee Food'),
            (df_merged['transactions_payment_type_name'] == 'Jiwa+'),
            (df_merged['transactions_payment_type_name'] == 'Grab Dine Out'),
            (df_merged['transactions_payment_type_name'] == 'Tokopedia Go'),
            (df_merged['transactions_payment_type_name'] == 'QPON')
        ]

        values = ['Go Food', 'Grab Food', 'Shopee Food', 'Jiwa+', 'Grab Dine Out', 'Tokopedia Go', 'Qpon']

        df_merged['sales_type'] = np.select(conditions, values, default='Walk In')

        df_merged.loc[df_merged['is_sjw'] == True, 'sales_type'] = 'Sejuta Jiwa'

        #finalize headers
        df_final_headers = df_merged[
            [
            'order_id', 'transaction_id', 'transaction_date',
            'order_reference', 'outlet_id', 'outlet_code', 'outlet_name', 'channel',
            'channel_detail', 'order_date', 'closed_date', 'status',
            'payment_status', 'fulfillment_status', 'transactions_type',
            'transactions_status', 'transactions_gateway', 'sales_type',
            'transactions_payment_type_name', 'notes', 'customer_id',
            'customer_first_name', 'customer_last_name', 'customer_phone_number',
            'customer_email', 'cashier_id', 'total_order_amount',
            'total_discount_amount', 'total_promotion_amount', 'subtotal',
            'total_tax_amount', 'total_amount', 'mdr'
            ]
        ]
        df_final_headers['load_data_at'] = datetime.datetime.now()
        df_final_headers.drop_duplicates(inplace=True)

        # Dibutuhkan proses grouping karena Mitra masih menggabungkan discount dalam promotion fields sehingga 1 order_details memiliki multiple rows

        def _join_text(sep):
            # gabung nilai teks unik (urutan asli), abaikan kosong
            return lambda s: sep.join(dict.fromkeys(str(v) for v in s if pd.notna(v) and str(v) != ''))

        # 1 row per order_detail_id: amount di-SUM, teks digabung -> merge ke details tidak fan-out
        if len(df_promo):
            df_grouping_promo = df_promo.groupby('order_detail_id', sort=False).agg(
                promotion_id=('promotion_id', _join_text(', ')),
                promotion_name=('promotion_name', _join_text('-')),
                promotion_type=('promotion_type', _join_text(', ')),
                promotion_amount=('promotion_amount', 'sum'),
            ).reset_index()
        else:
            df_grouping_promo = pd.DataFrame(columns=['order_detail_id', 'promotion_id', 'promotion_name', 'promotion_type', 'promotion_amount'])

        # discount juga di-group: sebelumnya di-merge mentah, detail dengan > 1 discount jadi > 1 row
        if len(df_discount):
            df_grouping_discount = df_discount.groupby('order_detail_id', sort=False).agg(
                discount_name=('discount_name', _join_text(', ')),
                discount_amount=('discount_amount', 'sum'),
            ).reset_index()
        else:
            df_grouping_discount = pd.DataFrame(columns=['order_detail_id', 'discount_name', 'discount_amount'])

        #merge promo and discount to order details
        df_merge_details_promo = df_details.merge(df_grouping_promo, on='order_detail_id', how='left')
        df_merge_details_promo_and_disc = df_merge_details_promo.merge(df_grouping_discount, on='order_detail_id', how='left')

        df_merge_details_promo_and_disc = df_merge_details_promo_and_disc[['order_id', 'order_detail_id', 'product_id', 'product_name',
            'product_generic_name', 'product_type', 'product_variant_id',
            'product_variant_name', 'product_modifier_id', 'product_modifier_name',
            'notes', 'sku', 'fulfillment_status', 'quantity',
            'base_price', 'total_order_amount', 'subtotal', 'additional_charge',
            'discount_percentage', 'discount_order_amount',
            'taxable', 'tax_percentage', 'tax_amount','type', 'promotion_id', 'promotion_name',
            'promotion_type', 'promotion_amount', 'discount_name',
            'discount_amount_y', 'modifier_product_sku']]

        #get and merge headers attribute 
        right_cols = ['order_id', 'transaction_date', 'order_reference',
            'outlet_id', 'outlet_code', 'outlet_name', 'channel', 'channel_detail',
            'order_date', 'closed_date', 'status', 'payment_status',
            'fulfillment_status', 'transactions_type', 'transactions_status', 'sales_type', 'transactions_payment_type_name',
            'notes', 'customer_id', 'customer_first_name', 'customer_last_name',
            'customer_phone_number', 'customer_email', 'cashier_id']

        df_details_2 = df_merge_details_promo_and_disc.merge(df_final_headers[right_cols], on = 'order_id', how = 'left') 


        #normalize combos 
        df_merge_details_and_combos = df_details_2.merge(df_product_combos, on= 'order_detail_id', how='left')

        df_merge_details_and_combos['bundling_id'] = ''
        for index, row in df_merge_details_and_combos.iterrows():
            if row['type'] == 'comboset':
                df_merge_details_and_combos.at[index, 'bundling_id'] = df_merge_details_and_combos.at[index, 'product_id_x']
                df_merge_details_and_combos.at[index, 'product_id_x'] = df_merge_details_and_combos.at[index, 'product_id_y']
                df_merge_details_and_combos.at[index, 'product_name_x'] = df_merge_details_and_combos.at[index, 'product_name_y']
                df_merge_details_and_combos.at[index, 'quantity_x'] = df_merge_details_and_combos.at[index, 'quantity_y']
                df_merge_details_and_combos.at[index, 'product_variant_name'] = df_merge_details_and_combos.at[index, 'variant_name']
                df_merge_details_and_combos.at[index, 'sku'] = df_merge_details_and_combos.at[index, 'product_sku'] 

        df_filtered = df_merge_details_and_combos[~((df_merge_details_and_combos['product_modifier_id'] != '') & (df_merge_details_and_combos['base_price'] == 0))]

        df_filtered = df_filtered[['order_id', 'transaction_date', 'order_reference', 'outlet_id',
            'outlet_code', 'outlet_name', 'channel', 'channel_detail', 'order_date',
            'closed_date', 'status', 'payment_status', 'transactions_type', 
            'transactions_status', 'sales_type', 'transactions_payment_type_name',
            'customer_id', 'customer_first_name', 'customer_last_name', 'customer_phone_number',
            'customer_email', 'cashier_id', 'order_detail_id', 'product_id_x', 'bundling_id','product_name_x',
            'product_generic_name','type', 'product_type', 'product_variant_id',
            'product_variant_name', 'product_modifier_id', 'product_modifier_name',
            'notes_x', 'sku', 'fulfillment_status_x', 'quantity_x', 'base_price',
            'total_order_amount','discount_name', 'discount_percentage', 'discount_order_amount',
            'discount_amount_y', 'promotion_id', 'promotion_name', 'promotion_type', 
            'promotion_amount', 'subtotal', 'taxable', 'tax_percentage', 
            'tax_amount', 'additional_charge', 'modifier_product_sku']]

        df_filtered.columns = ['order_id', 'transaction_date', 'order_reference', 'outlet_id',
            'outlet_code', 'outlet_name', 'channel', 'channel_detail', 'order_date',
            'closed_date', 'status', 'payment_status', 'transactions_type',
            'transactions_status', 'sales_type', 'transactions_payment_type_name',
            'customer_id', 'customer_first_name', 'customer_last_name',
            'customer_phone_number', 'customer_email', 'cashier_id',
            'order_detail_id', 'product_id', 'bundling_id', 'product_name',
            'product_generic_name', 'type', 'product_type', 'product_variant_id',
            'product_variant_name', 'product_modifier_id', 'product_modifier_name',
            'notes', 'sku', 'fulfillment_status', 'quantity', 'base_price',
            'total_order_amount', 'discount_name', 'discount_percentage', 'discount_order_amount',
            'discount_amount', 'promotion_id', 'promotion_name', 'promotion_type',
            'promotion_amount', 'subtotal', 'taxable', 'tax_percentage',
            'tax_amount', 'additional_charge', 'modifier_product_sku']

        df_filtered = pd.merge(df_filtered, df_products[['product_id', 'price']], on = 'product_id', how='left')

        # kalau tidak ada promo/diskon sama sekali, kolomnya bertipe object -> paksa angka sebelum fillna('')
        for col in ['promotion_amount', 'discount_amount', 'discount_order_amount']:
            df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce')

        # Normalize calculation
        numeric_columns = df_filtered.select_dtypes(include='number').columns
        string_columns = df_filtered.select_dtypes(include='object').columns

        df_filtered[numeric_columns] = df_filtered[numeric_columns].fillna(0)
        df_filtered[string_columns] = df_filtered[string_columns].fillna('')

        # Menghitung total_order_amount dengan kondisi berdasarkan keanggotaan product_id
        df_filtered['tax_rate'] = pd.to_numeric(df_filtered['tax_percentage'], errors='coerce').fillna(0) / 100
        df_filtered['divisor'] = 1 + df_filtered['tax_rate']
        
        is_combo = df_filtered['type'] == 'comboset'
        df_filtered['combo_share'] = 1.0
        if is_combo.any():
            combo = df_filtered[is_combo]
            key = [combo['order_detail_id'], combo['transactions_type']]
            weight = (combo['price'] * combo['quantity']).clip(lower=0)
            all_priced = (weight > 0).groupby(key).transform('all')
            share = np.where(all_priced,
                                weight / weight.groupby(key).transform('sum'),
                                1 / weight.groupby(key).transform('size'))
            df_filtered.loc[is_combo, 'combo_share'] = share
            df_filtered.loc[is_combo, 'total_order_amount'] = combo['total_order_amount'] * share

        def calculate_new_amount(group, amount_column):
            # 1 group = 1 order_detail dalam 1 transaksi -> tarif pajaknya sama (dari API)
            divisor = group['divisor'].iloc[0]
            if (group['type'] == 'comboset').all():
                # promo/diskon paket dibagi dengan bobot yang sama, supaya komponen murah tidak jadi minus
                return group[amount_column] * group['combo_share'] / divisor
            return group[amount_column] / len(group) / divisor

        detail_key = ['order_detail_id', 'transactions_type']

        df_filtered['discount_base'] = pd.to_numeric(df_filtered['discount_amount'], errors='coerce').fillna(0)

        df_filtered['discount_amount_new'] = df_filtered.groupby(detail_key)['discount_base'].transform(
            lambda x: calculate_new_amount(df_filtered.loc[x.index], 'discount_base')
        )

        df_filtered['promotion_amount_new'] = df_filtered.groupby(detail_key)['promotion_amount'].transform(
            lambda x: calculate_new_amount(df_filtered.loc[x.index], 'promotion_amount')
        )

        # Menghitung subtotal, tax_amount, dan total_amount
        df_filtered['subtotal'] = df_filtered['total_order_amount'] - df_filtered['discount_amount_new'] - df_filtered['promotion_amount_new']

        # -- versi lama (tarif dari list produk):
        # df_filtered['tax_amount'] = df_filtered.apply(
        #     lambda row: row['subtotal'] * 0.11 if row['product_id'] in product_id_sjw_sjo_v2 else row['subtotal'] * 0.1, axis=1
        # )
        df_filtered['tax_amount'] = df_filtered['subtotal'] * df_filtered['tax_rate']

        df_filtered['total_amount'] = df_filtered['subtotal'] + df_filtered['tax_amount']

        # Drop temporary columns
        df_filtered.drop(columns=['divisor', 'tax_rate', 'discount_base'], inplace=True)


        df_final_details = df_filtered[['order_id', 'transaction_date', 'order_reference', 'outlet_id',
            'outlet_code', 'outlet_name', 'channel', 'channel_detail', 'order_date',
            'closed_date', 'status', 'payment_status', 'transactions_type',
            'transactions_status', 'sales_type', 'transactions_payment_type_name',
            'customer_id', 'customer_first_name', 'customer_last_name',
            'customer_phone_number', 'customer_email', 'cashier_id',
            'order_detail_id', 'product_id', 'bundling_id', 'product_name',
            'product_generic_name', 'type', 'product_type', 'product_variant_id',
            'product_variant_name', 'product_modifier_id', 'product_modifier_name',
            'notes', 'sku', 'fulfillment_status', 'quantity', 'base_price',
            'total_order_amount', 'discount_name', 'discount_percentage',
            'discount_amount_new', 'promotion_id','promotion_name', 
            'promotion_type', 'promotion_amount_new', 'subtotal',
            'taxable', 'tax_percentage', 'tax_amount', 'additional_charge','total_amount', 'modifier_product_sku']]

        df_final_details.columns = ['order_id', 'transaction_date', 'order_reference', 'outlet_id',
            'outlet_code', 'outlet_name', 'channel', 'channel_detail', 'order_date',
            'closed_date', 'status', 'payment_status', 'transactions_type',
            'transactions_status', 'sales_type', 'transactions_payment_type_name',
            'customer_id', 'customer_first_name', 'customer_last_name',
            'customer_phone_number', 'customer_email', 'cashier_id',
            'order_detail_id', 'product_id', 'bundling_id', 'product_name',
            'product_generic_name', 'type', 'product_type', 'product_variant_id',
            'product_variant_name', 'product_modifier_id', 'product_modifier_name',
            'notes', 'sku', 'fulfillment_status', 'quantity', 'base_price',
            'total_order_amount', 'discount_name', 'discount_percentage',
            'discount_amount', 'promotion_id','promotion_name', 
            'promotion_type', 'promotion_amount', 'subtotal',
            'taxable', 'tax_percentage', 'tax_amount', 'additional_charge','total_amount', 'modifier_product_sku']

        df_final_details['load_data_at'] = datetime.datetime.now()

        df_final_details = df_final_details[['order_id', 'order_detail_id', 'type', 'bundling_id', 'product_id', 'sku', 'product_name',
                 'product_generic_name', 'base_price', 'quantity', 'total_order_amount', 'promotion_name',
                 'promotion_amount', 'discount_amount', 'subtotal', 'product_variant_id', 'product_variant_name', 
                 'product_modifier_id', 'product_modifier_name', 'notes', 'taxable', 'tax_percentage', 'tax_amount',
                 'additional_charge', 'total_amount', 'load_data_at', 'product_type', 'discount_name', 'discount_percentage',
                 'promotion_id',  'promotion_type', 'transaction_date', 'order_reference', 'outlet_id','outlet_code',
                 'outlet_name', 'channel', 'channel_detail', 'order_date', 'closed_date', 'status', 'payment_status',
                 'transactions_type', 'transactions_status', 'sales_type', 'transactions_payment_type_name', 'customer_id', 
                 'customer_first_name', 'customer_last_name', 'customer_phone_number', 'customer_email', 'cashier_id', 
                 'fulfillment_status', 'modifier_product_sku']]
        
        df_final_details['notes'] = df_final_details['notes'].astype(str).apply(lambda x: x.encode('ascii', 'ignore').decode('ascii'))

        # Mengupdate kolom 'sku' berdasarkan kondisi pada 'modifier_product_sku'

        df_final_details.loc[df_final_details['modifier_product_sku'] != '', 'sku'] = df_final_details['modifier_product_sku']
        df_final_details.drop(columns=['modifier_product_sku'], inplace=True)

    except Exception as e:
        import traceback; traceback.print_exc()
        raise  # jangan lanjut ke dump kalau transform gagal

### 7. Validate Total Data

In [ ]:
if len(get_data) > 0:
    try:
        if len(df_final_headers) > 0:
            total_amount_headers = df_final_headers['total_amount'].sum()
            if len(df_final_details['total_amount']) > 0:
                total_amount_details = df_final_details['total_amount'].sum()
                print(f"Total Amount Headers: {total_amount_headers}")
                print(f"Total Amount Details: {total_amount_details}")
                print(f"Accuracy Total Amount: {round((total_amount_details / total_amount_headers) * 100, 3)}")
                print(f"Total Order ID Duplicated: {len(df_final_headers[df_final_headers.duplicated('order_id')])}")
            else:
                print("tidak ada data details untuk backfilling")
        else:
            print("tidak ada data untuk backfilling")
    except Exception as e:
        print(f"An error accured : {e}")

### 8. Dump the Data and Send Email Notification

In [ ]:
import os
import smtplib, ssl
def send_email(subject, body):
    port = 587  # For starttls
    smtp_server = "smtp.gmail.com"
    sender_email = "saputra.christabel20@gmail.com"
    receiver_email = ["athens.jiwagroup@gmail.com"]
    password = os.getenv("GMAIL_SMTP_PASSWORD")

    message = f"""\
Subject: [iSeller MITRA] - {subject} Information (NEW)

{body}

This message is sent from Jupyter Notebook.

Best Regards,
DatO ( Data autOmation )"""

    context = ssl.create_default_context()
    with smtplib.SMTP(smtp_server, port) as server:
        server.ehlo()  # Can be omitted
        server.starttls(context=context)
        server.ehlo()  # Can be omitted
        server.login(sender_email, password)
        server.sendmail(sender_email, receiver_email, message)

if len(get_data) > 0:
    try: 
        if 'df_final_headers' in locals() and len(df_final_headers) > 0:
            total_amount_headers = df_final_headers['total_amount'].sum()

            if 'df_final_details' in locals() and len(df_final_details) > 0:
                total_amount_details = df_final_details['total_amount'].sum()

                # Dump data to the database
                engine = create_engine("mysql+pymysql://{user}:{pw}@{hostname}/{db}".format(
                    hostname=config_dl_dwh.hostname, 
                    user=config_dl_dwh.username,
                    pw=config_dl_dwh.password,
                    db=config_dl_dwh.db_resource))

                # Header + details dalam 1 transaksi: gagal di tengah -> rollback semua. Kalau tidak, order bisa
                # "sudah ada" di header tanpa details, dan run berikutnya (filter order) tidak akan pernah mengisinya.
                with engine.begin() as conn:
                    df_final_headers.to_sql('transactions_iseller_mitra', con=conn, if_exists='append', index=False)
                    df_final_details.to_sql('transactions_items_iseller_mitra', con=conn, if_exists='append', index=False)

                # Send success email (kalau ada panggilan API yang gagal, email PARTIAL dikirim di bawah)
                if not failed_calls:
                    send_email("Backfilling Data Transaction Success", "Hi! This is reporting Data for You!")
            
            else:
                # If df_final_details is empty
                print("Tidak ada data details untuk backfilling")
                if not failed_calls:
                    send_email("Backfilling Data Transaction: No Details", "Hi! There are no details data for backfilling.")
        else:
            # If df_final_headers is empty or not defined
            print("Tidak ada data headers untuk backfilling")
            if not failed_calls:
                send_email("Backfilling Data Transaction: No Headers and Details", "Hi! There are no headers and details data for backfilling.")
    except Exception as e:
        print(f"An error occurred: {e}")
        send_email("Backfilling Data Item Transaction: Error", f"An error occurred: {e}")
        raise
else:
    # If get_data is empty
    print("Tidak ada data untuk backfilling")
    if not failed_calls:
        send_email("Backfilling Data Transaction: No Data", "Hi! There are no data for backfilling.")

# Pengaman akhir: kalau ada panggilan API yang tetap gagal, run DIGAGALKAN (error merah) supaya tidak terlewat.
# Data yang berhasil sudah masuk di atas; run ulang hanya memasukkan order yang belum ada di Data Lake.
if failed_calls:
    detail = "\n".join(f"  {m[:200]}" for o, s, e, m in failed_calls[:30])
    msg = (f"{len(failed_calls)} panggilan API tetap gagal setelah retry. Data yang berhasil sudah di-dump, "
           f"data jendela berikut BELUM masuk (jalankan ulang notebook ini):\n{detail}")
    send_email(f"Backfilling Data Transaction: PARTIAL ({len(failed_calls)} API calls failed)", msg)
    raise Exception(msg)
